# Forward test: the volatility model on new trading days

[Open in Colab](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/forward_test.ipynb)

The frozen HAR-X model (HAR plus the long-run level, recent returns and volume) against HAR plus the long-run level, on the S&P 500 and S&P MidCap 400, scoring only trading days from 2026-11-19. The rule is `protocols/har_x_forward_v1.yaml`, registered 2026-11-18.

**How to use it:** every few months, Runtime → Run all, then save to GitHub or paste the output. It downloads both universes to yesterday's close (about 30 minutes), so it needs your Alpaca keys in Colab Secrets. There is nothing to set.

**What it prints:** a progress report, labelled interim, until 500 trading days (about November 2028), when it gives the one verdict. The first useful run is after about 125 trading days, around May 2027. Interim reports never stop or change the test, and the model is never retuned.

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/entropy-gate-pipeline' if IN_COLAB else os.environ.get('PORTFOLIO_REPO', os.getcwd())
if IN_COLAB:
    if os.path.isdir(f'{REPO}/.git'):
        !cd {REPO} && git pull -q --ff-only
    else:
        !git clone -q https://github.com/Gaire-commits/entropy-gate-pipeline.git {REPO}
    %cd {REPO}
    !git log --oneline -1
    from google.colab import userdata
    try:
        with open('.env', 'w') as f:
            f.write(f"APCA_API_KEY_ID={userdata.get('APCA_API_KEY_ID')}\nAPCA_API_SECRET_KEY={userdata.get('APCA_API_SECRET_KEY')}\n")
        print('keys written to a local .env (never printed, never committed)')
    except Exception as exc:
        print('No Alpaca keys in Colab Secrets:', type(exc).__name__, '- add them from the key icon, then run this cell again.')
else:
    os.chdir(REPO)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
sys.path[:0] = [REPO, f'{REPO}/scripts']

from pathlib import Path
REF = os.environ.get('RESERVED_REF', '/content/reserved/reference')
WORK = os.environ.get('RESERVED_WORK', '/content/reserved')
DRIVE = '/content/drive/MyDrive/entropy-gate-pipeline-data/data'
Path(REF).mkdir(parents=True, exist_ok=True)

In [ ]:
import datetime
from pathlib import Path
# Up to yesterday: the Alpaca plan refuses SIP data from the last 15 minutes, and only completed sessions count.
END = (datetime.date.today() - datetime.timedelta(days=1)).isoformat()
ok = True
for NAME, LIST in (('sp500', 'universe/sp500.csv'), ('sp400', 'universe/sp400.csv')):
    OUT = f'/content/forward/{NAME}'
    !python scripts/fetch_reference.py --universe {LIST} --start 2016-01-04 --end {END} --out {OUT} --parts daily,corporate_actions,meta,auctions --force 2>&1 | tail -2
    if not Path(f'{OUT}/daily/daily_sip_raw.parquet').exists() or not Path(f'{OUT}/auctions.parquet').exists():
        print(f'STOPPED: the {NAME} download did not finish (see the lines above); nothing was scored.')
        ok = False
        break
    !python scripts/official_prices.py --reference {OUT} 2>&1 | tail -1
if ok:
    !python scripts/har_x_forward.py --sp500 /content/forward/sp500 --sp400 /content/forward/sp400 2>&1 | grep -v Warning